In [19]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
from langchain.chat_models import ChatOpenAI
from langgraph.graph import StateGraph, START
from typing import TypedDict, Any, List, Dict
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
import openai
import os

# TypedDict for agent state
class AgentState(TypedDict):
    instruction: str
    components: List[Dict[str, Any]]
    plan: str
    model_code: str

# 1) Load embeddings and index
embedder = OpenAIEmbeddings()
index = FAISS.load_local(
    "../faiss_index",
    embedder,
    allow_dangerous_deserialization=True
)

# 2) LLM client
openai.api_key = os.getenv("XAI_API_KEY")
llm = ChatOpenAI(
    model_name="grok-3-beta",
    api_key=openai.api_key,
    base_url="https://api.x.ai/v1",
    temperature=0
)

# 3) RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.as_retriever(),
    return_source_documents=True
)

## 4) Helper: parse Modelica component definitions
def parse_components(docs: List[Any]) -> List[Dict[str, str]]:
    components = []
    for doc in docs:
        name = doc.metadata.get('source', 'Unknown')
        components.append({'name': name, 'code': doc.page_content})
    return components

# 5) Sync wrapper for retrieve_components
def retrieve_components_sync(state: AgentState) -> AgentState:
    loop = asyncio.get_event_loop()
    result = loop.run_until_complete(
        qa.acall(state['instruction'])  # returns {'result', 'source_documents'}
    )
    state['components'] = parse_components(result['source_documents'])
    return state

# 6) Async nodes for plan and generate using ainvoke
async def plan_model_step(state: AgentState) -> AgentState:
    prompt = (
        "Given these Modelica components:\n"
        f"{state['components']}\n"
        f"And the user wants: {state['instruction']}\n"
        "Generate a step-by-step plan to assemble the Modelica model."
    )
    # Use ainvoke (takes text prompt) instead of deprecated apredict
    state['plan'] = await llm.ainvoke(prompt)
    return state

async def generate_model_code(state: AgentState) -> AgentState:
    prompt = (
        "Using the plan below and the components provided, produce the full Modelica model code:\n"
        f"Plan:\n{state['plan']}\nComponents:\n{state['components']}\n"
    )
    state['model_code'] = await llm.ainvoke(prompt)
    return state

# 7) Build and compile LangGraph
graph = StateGraph(AgentState)
graph.add_node("retrieve_components", retrieve_components_sync)
graph.add_node("plan_model_step", plan_model_step)
graph.add_node("generate_model_code", generate_model_code)
graph.add_edge(START, "retrieve_components")
graph.add_edge("retrieve_components", "plan_model_step")
graph.add_edge("plan_model_step", "generate_model_code")
compiled = graph.compile()

# 8) Execution example (Jupyter)

# (A) 環境変数で LangChain トレーシングをオフにする方法
# export LANGCHAIN_HANDLER=None
# export LANGCHAIN_TRACING_V2=False

initial: AgentState = {
    "instruction": "Construct room air conditioner model.",
    "components": [],
    "plan": "",
    "model_code": ""
}

# (B) コールバック指定なしで実行
# 'callbacks' のキーワード引数は Pregel.astream で未対応
# トラッキングをオフにしたうえで、シンプルに呼び出します
result = await compiled.ainvoke(
    initial
)
#print(result["model_code"])

In [18]:
import re
from rich.console import Console
from rich.syntax import Syntax

# 1) AIMessage から純粋な文字列を取り出す
raw_msg = result["model_code"]
if hasattr(raw_msg, "content"):
    raw = raw_msg.content
else:
    raw = str(raw_msg)

# 2) ```modelica``` ブロックだけ抜き出す
#    バックスラッシュは Python リテラル内でエスケープしないほうが読みやすいので
match = re.search(r"```modelica\n(.+?)```", raw, flags=re.DOTALL)
if match:
    code = match.group(1).strip()
else:
    # ブロックが見つからなければ全体をコードと見なす
    code = raw

# 3) Rich でシンタックスハイライト
console = Console()
syntax = Syntax(code, "modelica", line_numbers=True)
console.print(syntax)

   1 model RoomAirConditioner                                                                                      
   2   "Model of a room air conditioner using a reversible heat pump in cooling mode"                              
   3                                                                                                               
   4   // Import necessary libraries                                                                               
   5   import AixLib;                                                                                              
   6                                                                                                               
   7   // Room model                                                                                               
   8   AixLib.Fluid.HeatPumps.ModularReversible.Examples.BaseClasses.PartialOneRoomRadiator room(                  
   9     redeclare package Medium = AixLib.Media.Air,                                                              
  10     V=100 "Room volume in m3 (e.g., 5m x 5m x 4m)",                                                           
  11     hConWin=2.7 "Convective heat transfer coefficient for windows",                                           
  12     hConWall=1.7 "Convective heat transfer coefficient for walls",                                            
  13     hRad=5 "Radiative heat transfer coefficient",                                                             
  14     AWin=5 "Window area in m2",                                                                               
  15     AWall=40 "Wall area in m2",                                                                               
  16     AInt=20 "Internal surface area in m2",                                                                    
  17     epsWin=0.9 "Emissivity of window",                                                                        
  18     epsWall=0.9 "Emissivity of wall",                                                                         
  19     Q_flow_int=200 "Internal heat gains (e.g., occupants, equipment) in W",                                   
  20     T_start=301.15 "Initial room temperature (28°C)"                                                          
  21   ) "Thermal model of the room";                                                                              
  22                                                                                                               
  23   // Heat pump model configured for cooling                                                                   
  24   AixLib.Fluid.HeatPumps.ModularReversible.Examples.Modular_OneRoomRadiator heatPump(                         
  25     redeclare package Medium1 = AixLib.Media.Water "Condenser side medium (water or air simulation)",         
  26     redeclare package Medium2 = AixLib.Media.Water "Evaporator side medium (water or air simulation)",        
  27     redeclare AixLib.Fluid.HeatPumps.Data.ScrollWaterToWater.Heating.ClimateMaster_TMW036_12kW_4_90COP_R410A d
  28     use_rev=true "Enable reversible operation (cooling and heating)",                                         
  29     isHea=false "Set to cooling mode (false for cooling, true for heating)",                                  
  30     QCon_flow_nominal=12000 "Nominal condenser heat flow rate (W)",                                           
  31     QEva_flow_nominal=-[38;2;174;12

In [16]:
!uv add rich

Resolved 182 packages in 617ms                                       
Installed 3 packages in 15ms                                     
 + markdown-it-py==3.0.0
 + mdurl==0.1.2
 + rich==14.0.0


In [21]:
result['plan']

AIMessage(content='To construct a room air conditioner model using the provided Modelica components from the AixLib library, we will focus on leveraging relevant heat pump models and associated components for a typical air conditioning setup. Since the provided components are primarily related to water-to-water heat pumps, we will adapt the approach to simulate an air conditioning system by considering air-to-air or air-to-water configurations if necessary, and using modular reversible heat pump models for flexibility. Below is a step-by-step plan to assemble the Modelica model for a room air conditioner.\n\n### Step-by-Step Plan to Construct a Room Air Conditioner Model\n\n#### Step 1: Define the Objective and Scope\n- **Objective**: Create a Modelica model for a room air conditioner to cool a single room.\n- **Scope**: The model should include a heat pump (acting as an air conditioner in cooling mode), a room model for thermal load simulation, and basic control logic for temperature 

In [26]:
import re

def convert_ai_message_to_markdown(ai_message_text: str, output_filename: str = "room_air_conditioner_model.md"):
    """
    Convert a structured AI plan into a Markdown-formatted document and save it.
    """
    # Clean up extra newlines
    content = ai_message_text.strip()
    content = re.sub(r"\n{3,}", "\n\n", content)  # Max double newline
    
    # Replace markdown-style headers with proper markdown
    content = re.sub(r"^### (.*?)$", r"### \1", content, flags=re.MULTILINE)
    content = re.sub(r"^#### (.*?)$", r"#### \1", content, flags=re.MULTILINE)
    content = re.sub(r"^## (.*?)$", r"## \1", content, flags=re.MULTILINE)

    # Optional: Emphasize bold/italic in a more readable way if needed
    content = re.sub(r"\*\*(.*?)\*\*", r"**\1**", content)  # Already markdown

    # Save to file
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(content)
    
    print(f"✅ Markdown file saved as: {output_filename}")

# 例：AI Messageの中身を変数に代入して渡す
ai_generated_text = result["plan"]

# 実行
convert_ai_message_to_markdown(ai_generated_text)

AttributeError: 'AIMessage' object has no attribute 'strip'